In [40]:
print('hi')

hi


In [41]:
import numpy as np
import networkx as nx
import pandas as pd
import random
from collections import Counter, defaultdict

In [42]:
np.random.seed(42)
random.seed(42)

# --- SBM parameters ---
num_blocks = 10
block_size = 10
n = num_blocks * block_size

# block membership
labels = np.repeat(np.arange(num_blocks), block_size)

# high homophily: strong within-block, weak between-block
p_in = 0.8
p_out = 0.1

P = np.full((num_blocks, num_blocks), p_out)
np.fill_diagonal(P, p_in)

# --- generate SBM adjacency matrix ---
A = np.zeros((n, n))

for i in range(n):
    for j in range(i+1, n):
        pi = P[labels[i], labels[j]]
        if np.random.rand() < pi:
            A[i, j] = 1
            A[j, i] = 1

G = nx.from_numpy_array(A)

print(A.shape)

(100, 100)


In [43]:
df = pd.DataFrame({
    "node_id": np.arange(n),
    "block": labels
})

df["degree"] = dict(G.degree()).values()

In [44]:
opinions_all = [1, 2, 3, 4]

# 1. assign a dominant opinion to each block
block_dominant = {
    b: np.random.choice(opinions_all)
    for b in range(num_blocks)
}

opinions = []

for i in range(n):
    b = df.loc[i, "block"]
    dominant = block_dominant[b]

    # build probability vector
    probs = np.full(4, 0.5 / 3)  # 3 non-dominant opinions
    probs[dominant - 1] = 0.5    # index shift (opinions are 1–4)

    # sample
    opinions.append(np.random.choice(opinions_all, p=probs))

df["opinion"] = opinions

In [45]:
print(df)

    node_id  block  degree  opinion
0         0      0      19        3
1         1      0      16        2
2         2      0      15        2
3         3      0      23        2
4         4      0      21        3
..      ...    ...     ...      ...
95       95      9      14        2
96       96      9      13        4
97       97      9      18        2
98       98      9      15        1
99       99      9      18        3

[100 rows x 4 columns]


In [46]:
def rds_with_tracking(G, seeds, sample_size=50, max_recruits=2):
    import random
    
    sampled = set(seeds)
    queue = [(s, 0) for s in seeds]  # (node, wave)
    
    records = []
    
    # initialize seeds
    for s in seeds:
        records.append({
            "node_id": s,
            "recruiter": None,
            "wave": 0
        })
    
    referrer_to_recruits = defaultdict(list)
    
    while queue and len(sampled) < sample_size:
        node, wave = queue.pop(0)
        
        neighbors = list(G.neighbors(node))
        random.shuffle(neighbors)
        
        recruits = 0
        
        for nb in neighbors:
            if nb not in sampled:
                sampled.add(nb)
                
                records.append({
                    "node_id": nb,
                    "recruiter": node,
                    "wave": wave + 1
                })

                referrer_to_recruits[node].append(nb)
                
                queue.append((nb, wave + 1))
                
                recruits += 1
                if recruits >= max_recruits or len(sampled) >= sample_size:
                    break
    
    return pd.DataFrame(records), referrer_to_recruits

seeds = random.sample(range(n), 5)

rds_df, referrer_to_recruits = rds_with_tracking(G, seeds, sample_size=30, max_recruits=2)

rds_df = rds_df.merge(df[["node_id", "opinion", "degree", "block"]], on="node_id", how="left")
print(rds_df)

    node_id  recruiter  wave  opinion  degree  block
0        81        NaN     0        2      17      8
1        14        NaN     0        1      17      1
2         3        NaN     0        2      23      0
3        94        NaN     0        4      11      9
4        35        NaN     0        4      14      3
5        29       81.0     1        2      13      2
6        39       81.0     1        1      19      3
7        30       14.0     1        1      19      3
8        58       14.0     1        2      20      5
9         4        3.0     1        3      21      0
10       22        3.0     1        1      22      2
11       11       94.0     1        4      19      1
12       91       94.0     1        3      20      9
13        5       35.0     1        2      12      0
14       71       35.0     1        2      20      7
15       23       29.0     2        1      15      2
16       25       29.0     2        4      17      2
17       33       39.0     2        3      19 

In [47]:
print(referrer_to_recruits)

defaultdict(<class 'list'>, {81: [29, 39], 14: [30, 58], 3: [4, 22], 94: [11, 91], 35: [5, 71], 29: [23, 25], 39: [33, 31], 30: [37, 0], 58: [66, 69], 4: [49, 9], 22: [27, 28], 11: [12, 13], 91: [59]})


In [48]:
def get_distribution(series):
    return series.value_counts(normalize=True).sort_index()

pop_dist = get_distribution(df["opinion"])
print("True population distribution:\n", pop_dist, "\n")

True population distribution:
 opinion
1    0.21
2    0.34
3    0.18
4    0.27
Name: proportion, dtype: float64 



In [49]:
def vh_estimator(rds_df):
    weights = 1 / rds_df["degree"]
    
    weighted_counts = (
        rds_df
        .assign(weight=weights)
        .groupby("opinion")["weight"]
        .sum()
    )

    return (weighted_counts / weighted_counts.sum()).sort_index()

sample_dist = get_distribution(rds_df["opinion"])
print("Naive sample distribution:\n", sample_dist, "\n")

vh_dist = vh_estimator(rds_df)
print("VH estimate:\n", vh_dist, "\n")

Naive sample distribution:
 opinion
1    0.200000
2    0.300000
3    0.233333
4    0.266667
Name: proportion, dtype: float64 

VH estimate:
 opinion
1    0.189464
2    0.296641
3    0.223912
4    0.289983
Name: weight, dtype: float64 



In [50]:
def total_variation(p, q):
    idx = sorted(set(p.index).union(q.index))
    p = p.reindex(idx, fill_value=0)
    q = q.reindex(idx, fill_value=0)
    return 0.5 * np.abs(p - q).sum()

In [51]:
recruiter_map = {}

for r, recruits in referrer_to_recruits.items():
    for u in recruits:
        recruiter_map[u] = r

In [52]:
def vh_with_homophily_adjustment(rds_df, recruiter_map, alpha=0.5):
    weights = []
    
    for _, row in rds_df.iterrows():
        node = row["node_id"]
        base_weight = 1 / row["degree"]
        
        recruiter = recruiter_map.get(node, None)
        
        if recruiter is None:
            weights.append(base_weight)  # seed
            continue
        
        # compare blocks
        if df.loc[node, "block"] == df.loc[recruiter, "block"]:
            weights.append(alpha * base_weight)
        else:
            weights.append(base_weight)
    
    rds_df = rds_df.copy()
    rds_df["weight"] = weights
    
    weighted_counts = (
        rds_df
        .groupby("opinion")["weight"]
        .sum()
    )
    
    return (weighted_counts / weighted_counts.sum()).sort_index()

sample_dist = get_distribution(rds_df["opinion"])
adj_dist = vh_with_homophily_adjustment(rds_df, recruiter_map, alpha=0.2)
pop_dist = get_distribution(df["opinion"])

print("Sample distribution:\n", sample_dist, "\n")
print("Adjusted VH estimate:\n", adj_dist, "\n")
print("True population distribution:\n", pop_dist, "\n")

print("TV (sample vs population):", total_variation(sample_dist, pop_dist))
print("TV (VH vs population):", total_variation(vh_dist, pop_dist))
print("TV (Adjusted vs population):", total_variation(adj_dist, pop_dist))

Sample distribution:
 opinion
1    0.200000
2    0.300000
3    0.233333
4    0.266667
Name: proportion, dtype: float64 

Adjusted VH estimate:
 opinion
1    0.197761
2    0.362476
3    0.135425
4    0.304338
Name: weight, dtype: float64 

True population distribution:
 opinion
1    0.21
2    0.34
3    0.18
4    0.27
Name: proportion, dtype: float64 

TV (sample vs population): 0.05333333333333336
TV (VH vs population): 0.0638948766696673
TV (Adjusted vs population): 0.056814418928585256
